Checklist to start training models on a windows machine

In [1]:
import platform 
import subprocess 
import sys

print("Python: ", sys.version)
print("Platform: ", platform.system())
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)




Python:  3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Platform:  Windows
Sun Jul 19 08:58:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:02:00.0  On |                  N/A |
| 32%   38C    P0             28W /  300W |    2237MiB /  16303MiB |      0%      Default |
|                              

In [2]:
import shutil 
nvcc = shutil.which("nvcc")
print("nvcc path:", nvcc)
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout if nvcc else "Toolkit not installed or not on PATH")

nvcc path: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.1\bin\nvcc.EXE
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Nov__7_19:25:04_Pacific_Standard_Time_2025
Cuda compilation tools, release 13.1, V13.1.80
Build cuda_13.1.r13.1/compiler.36836380_0



Use below script to check if cuda toolkit and nvidia driver are compatible

In [3]:
import re
import shutil
import subprocess

def run(command):
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode:
        raise RuntimeError(completed.stderr.strip() or f"Command failed: {command}")
    return completed.stdout

def parse_cuda_version(text, pattern, source):
    match = re.search(pattern, text)
    if not match:
        raise RuntimeError(f"Could not find a CUDA version in {source} output.")
    return tuple(map(int, match.group(1).split(".")))

assert shutil.which("nvcc"), "nvcc is unavailable; install the Toolkit or fix PATH."
smi_output = run(["nvidia-smi"])
nvcc_output = run(["nvcc", "--version"])

driver_cuda = parse_cuda_version(smi_output, r"CUDA Version:\s*(\d+\.\d+)", "nvidia-smi")
toolkit_cuda = parse_cuda_version(nvcc_output, r"release\s+(\d+\.\d+)", "nvcc")

print("Driver CUDA compatibility ceiling:", ".".join(map(str, driver_cuda)))
print("Installed CUDA Toolkit:", ".".join(map(str, toolkit_cuda)))

if toolkit_cuda <= driver_cuda:
    print("PASS: the installed Toolkit is within the driver's reported CUDA compatibility ceiling.")
else:
    print(
        "FAIL: the Toolkit is newer than the driver's reported CUDA compatibility ceiling. "
        "Update the NVIDIA driver or install a Toolkit version no newer than that ceiling."
    )

Driver CUDA compatibility ceiling: 13.1
Installed CUDA Toolkit: 13.1
PASS: the installed Toolkit is within the driver's reported CUDA compatibility ceiling.
